In [ ]:
import pandas as pd
import altair as alt

# Cargamos el dataset de demanda ya limpio y particionado en parquet en la fase de oro
df_oro = pd.read_parquet('data/gold/dataset_demanda_parquet')

# Agrupamos los datos a nivel diario calculando medias y máximas
df_diario = df_oro.groupby(pd.to_datetime(df_oro['fecha']).dt.date).agg({
    'demanda_mw': 'mean',
    'temperatura_maxima': 'max',
    'tipo_dia': 'first'
}).reset_index()

# Aseguramos formatos de datos importantes tras la agrupación
df_diario['fecha'] = pd.to_datetime(df_diario['fecha'])
df_diario['tipo_dia'] = df_diario['tipo_dia'].astype(int)

# Aplicamos el diccionario analítico de los tipos de días
diccionario_dias = {
    0: '0 - Laborable', 
    1: '1 - Festivo Estándar', 
    2: '2 - Festivo Crítico'
}
df_diario['categoria_visual'] = df_diario['tipo_dia'].map(diccionario_dias)

# Creamos un selector de intervalo exclusivo para el eje X del gráfico de demanda energética
selector_fecha = alt.selection_interval(encodings=['x'])

# CONSTRUCCIÓN DE LOS GRÁFICOS

# Demanda energética a lo largo del tiempo
# Gráfico A.1: Evolución Temporal en la demanda (Azul)
linea_base = alt.Chart(df_diario).mark_line(
    color='#1f77b4', strokeWidth=3, opacity=0.3
).encode(
    x=alt.X('fecha:T', title='Fecha'),
    y=alt.Y('demanda_mw:Q', title='Demanda Media Diaria (MW)', scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip('fecha:T', title='Fecha'), 
        alt.Tooltip('demanda_mw:Q', title='Demanda Real (MW)', format='.0f')
    ]
)

# Gráfico A.2: Media Móvil de 7 días para mostrar la tendencia (Rojo)
linea_tendencia = alt.Chart(df_diario).mark_line(
    color='#d62728', strokeWidth=1.5
).transform_window(
    media_movil='mean(demanda_mw)',
    sort=[alt.SortField('fecha')],
    frame=[-7, 0] # Calcula la media usando los 7 días anteriores
).encode(
    x='fecha:T',
    y='media_movil:Q',
    tooltip=[
        alt.Tooltip('fecha:T', title='Fecha'),
        alt.Tooltip('media_movil:Q', title='Tendencia (MW)', format='.0f')
    ]
)

# Superponemos las dos capas (+) y le enchufamos tu selector de fechas
grafico_linea = (linea_base + linea_tendencia).properties(
    width=750,
    height=250,
    title="Consumo Diario (Azul) vs. Tendencia Real (Rojo)"
).add_params(
    selector_fecha
)

# Gráfico B: Impacto de las temperaturas en la demanda energética
grafico_clima = alt.Chart(df_diario).mark_circle(size=60, opacity=0.6, color='#ff7f0e').encode(
    x=alt.X('temperatura_maxima:Q', title='Temperatura Máxima (ºC)', scale=alt.Scale(zero=False)),
    y=alt.Y('demanda_mw:Q', title='Demanda Media (MW)', scale=alt.Scale(zero=False)),
    tooltip=[
        alt.Tooltip('fecha:T', title='Fecha'),
        alt.Tooltip('temperatura_maxima:Q', title='Temp. Máx (ºC)', format='.1f'),
        alt.Tooltip('demanda_mw:Q', title='Demanda (MW)', format='.0f')
    ]
).properties(
    width=340,
    height=280,
    title="Cómo Afecta la temperatura al Consumo"
).transform_filter(
    selector_fecha
)

# Gráfico C: Impacto de los días festivos y laborales en la demanda energética
grafico_festivos = alt.Chart(df_diario).mark_bar(
    cornerRadiusTopLeft=5, 
    cornerRadiusTopRight=5
).encode(
    x=alt.X('categoria_visual:N', title='', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('mean(demanda_mw):Q', title='Demanda Media (MW)'),
    color=alt.Color(
        'categoria_visual:N', 
        legend=None, 
        scale=alt.Scale(
            domain=['0 - Laborable', '1 - Festivo Estándar', '2 - Festivo Crítico'],
            range=['#1f77b4', '#ff7f0e', '#d62728']
        )
    ),
    tooltip=[
        alt.Tooltip('categoria_visual:N', title='Tipo de Día'), 
        alt.Tooltip('mean(demanda_mw):Q', title='Demanda Media (MW)', format='.0f')
    ]
).properties(
    width=340,
    height=280,
    title="Consumo en Días Laborables vs. Festivos"
).transform_filter(
    selector_fecha
)

# Composición final del Dashboard
dashboard_final = grafico_linea & (grafico_clima | grafico_festivos)

# Mostramos el cuadro de mando en el cuaderno
dashboard_final.display()

alt.VConcatChart(...)